## Libraries & Organizing

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

## Datasets

In [ ]:
# ============================================================
# Load datasets
# ============================================================

dataset_llama = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs.csv")
dataset_qwen = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs_qwen.csv")


## Regression

In [ ]:

# ============================================================
# Settings
# ============================================================

car_vars = [
    "car_immediate",
    "car_medium",
    "car_long"
]

fe_specs = {
    "FIRM": "C(permno)",
    "IND": "C(gind)"
}

output_dir = DATA_PROCESSED / "regressions"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# Group rare qualitative LLM categories
# ============================================================

qual_cols = [
    "main_focus",
    "managerial_horizon",
]

def group_rare_categories(df, cols, min_count=30):
    df = df.copy()

    for col in cols:
        df[col] = df[col].astype(str).str.strip()

        counts = df[col].value_counts()
        keep_categories = counts[counts >= min_count].index

        df[col + "_grouped"] = df[col].where(
            df[col].isin(keep_categories),
            "Other"
        )

    return df


dataset_llama = group_rare_categories(
    dataset_llama,
    qual_cols,
    min_count=30
)

dataset_qwen = group_rare_categories(
    dataset_qwen,
    qual_cols,
    min_count=30
)

datasets = {
    "llama": dataset_llama,
    "qwen": dataset_qwen
}


In [ ]:



# ============================================================
# Clean regression output
# ============================================================

def clean_regression_output(model, title):
    params = model.params
    bse = model.bse
    pvals = model.pvalues

    rows = []

    for var in params.index:
        # Hide firm/industry FE only
        if var.startswith("C(permno)") or var.startswith("C(gind)"):
            continue

        rows.append({
            "Variable": var,
            "Coef.": params[var],
            "Std.Err.": bse[var],
            "P>|t|": pvals[var]
        })

    out = pd.DataFrame(rows)

    text = []
    text.append(title)
    text.append("-" * 80)
    text.append(out.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    text.append("")
    text.append(f"N: {int(model.nobs)}")
    text.append(f"R-squared: {model.rsquared:.4f}")
    text.append(f"Adj. R-squared: {model.rsquared_adj:.4f}")
    text.append("")
    text.append("Note: Firm/industry fixed effects are included where specified but not reported.")
    text.append("Year effects and grouped qualitative LLM categories are reported.")
    text.append("Standard errors are clustered by firm (PERMNO).")

    return "\n".join(text)


# ============================================================
# Regression function
# ============================================================

def run_regressions(df, model_name, car_var, fe_name, fe_term):
    df = df.copy()

    # Contextualized FLI
    df["context_score"] = df[
        [
            "specificity",
            "economic_substance",
            "certainty"
        ]
    ].mean(axis=1)

    df["contextual_fli"] = (
        df["forward_looking_intensity"]
        * df["context_score"]
    )



    # ========================================================
    # Specification 0A: LLM FLI only + controls only
    # No year FE, no firm/industry FE
    # ========================================================

    vars_fli_controls_only = [
        car_var,
        "forward_looking_intensity",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "gind"
    ]

    reg_fli_controls_only = df[vars_fli_controls_only].dropna().copy()

    formula_fli_controls_only = f"""
    {car_var} ~
    forward_looking_intensity +
    bm +
    log_word_count +
    log_assets +
    {fe_term}
    """

    model_fli_controls_only = smf.ols(
        formula=formula_fli_controls_only,
        data=reg_fli_controls_only
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_fli_controls_only["permno"]}
)

    # ========================================================
    # Specification 1: LLM FLI only + controls
    # ========================================================

    vars_fli_only = [
        car_var,
        "forward_looking_intensity",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind"
    ]

    reg_fli = df[vars_fli_only].dropna().copy()

    formula_fli_only = f"""
    {car_var} ~
    forward_looking_intensity +
    specificity +
    economic_substance +
    tone +
    certainty +
    bm +
    log_word_count +
    log_assets +
    C(year) +
    {fe_term}
    """

    model_fli_only = smf.ols(
        formula=formula_fli_only,
        data=reg_fli
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_fli["permno"]}
    )

    # ========================================================
    # Specification 2: Additive LLM dimensions WITHOUT qualitative variables
    # ========================================================

    vars_additive_noqual = [
        car_var,
        "forward_looking_intensity",
        "dict_score",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind"
    ]

    reg_add_noqual = df[vars_additive_noqual].dropna().copy()

    formula_additive_noqual = f"""
    {car_var} ~
    forward_looking_intensity +
    dict_score +
    specificity +
    economic_substance +
    tone +
    certainty +
    bm +
    log_word_count +
    log_assets +
    C(year) +
    {fe_term}
    """

    model_additive_noqual = smf.ols(
        formula=formula_additive_noqual,
        data=reg_add_noqual
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_add_noqual["permno"]}
    )

    # ========================================================
    # Specification 3: Additive LLM dimensions WITH qualitative variables
    # ========================================================

    vars_additive_qual = [
        car_var,
        "forward_looking_intensity",
        "dict_score",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind",
        "main_focus_grouped",
        "managerial_horizon_grouped",
    ]

    reg_add_qual = df[vars_additive_qual].dropna().copy()

    formula_additive_qual = f"""
    {car_var} ~
    forward_looking_intensity +
    dict_score +
    specificity +
    economic_substance +
    tone +
    certainty +
    bm +
    log_word_count +
    log_assets +
    C(main_focus_grouped) +
    C(managerial_horizon_grouped) +
    C(year) +
    {fe_term}
    """

    model_additive_qual = smf.ols(
        formula=formula_additive_qual,
        data=reg_add_qual
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_add_qual["permno"]}
    )

    # ========================================================
    # Specification 4: Contextualized FLI WITH qualitative variables
    # ========================================================

    vars_contextual_qual = [
        car_var,
        "contextual_fli",
        "dict_score",
        "tone",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind",
        "main_focus_grouped",
        "managerial_horizon_grouped",
    ]

    reg_ctx_qual = df[vars_contextual_qual].dropna().copy()

    formula_contextual_qual = f"""
    {car_var} ~
    contextual_fli +
    dict_score +
    tone +
    bm +
    log_word_count +
    log_assets +
    C(main_focus_grouped) +
    C(managerial_horizon_grouped) +
    C(year) +
    {fe_term}
    """

    model_contextual_qual = smf.ols(
        formula=formula_contextual_qual,
        data=reg_ctx_qual
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_ctx_qual["permno"]}
    )

    # ========================================================
    # Save output
    # ========================================================

    file_name = f"regression_{car_var}_{fe_name}_{model_name}.txt"
    file_path = output_dir / file_name

    with open(file_path, "w") as f:
        f.write(f"Model: {model_name.upper()}\n")
        f.write(f"Dependent variable: {car_var}\n")
        f.write(f"Fixed effects: {fe_name}\n")
        f.write("=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_fli_controls_only,
            "SPECIFICATION 0: LLM FLI + CONTROLS ONLY"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_fli_only,
            "SPECIFICATION 1: LLM FLI + CONTEXTUAL DIMENSIONS"
        ))
        f.write("\n\n" + "=" * 80 + "\n\n") 

        f.write(clean_regression_output(
            model_additive_noqual,
            "SPECIFICATION 2: ADDITIVE LLM DIMENSIONS WITHOUT QUALITATIVE VARIABLES"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_additive_qual,
            "SPECIFICATION 3: ADDITIVE LLM DIMENSIONS WITH GROUPED QUALITATIVE VARIABLES"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_contextual_qual,
            "SPECIFICATION 4: CONTEXTUALIZED FLI WITH GROUPED QUALITATIVE VARIABLES"
        ))

    return (
        model_fli_controls_only,
        model_fli_only,
        model_additive_noqual,
        model_additive_qual,
        model_contextual_qual
)


# ============================================================
# Run all regressions
# ============================================================

results = {}

for model_name, df in datasets.items():
    for car_var in car_vars:
        for fe_name, fe_term in fe_specs.items():

            print(f"Running {model_name} | {car_var} | {fe_name}")

            (
            fli_controls_only_model,
            fli_model,
            add_noqual_model,
            add_qual_model,
            ctx_qual_model
        ) = run_regressions(
                df=df,
                model_name=model_name,
                car_var=car_var,
                fe_name=fe_name,
                fe_term=fe_term
            )

            results[(model_name, car_var, fe_name, "fli_controls_only")] = fli_controls_only_model
            results[(model_name, car_var, fe_name, "fli_only")] = fli_model
            results[(model_name, car_var, fe_name, "additive_noqual")] = add_noqual_model
            results[(model_name, car_var, fe_name, "additive_qual")] = add_qual_model
            results[(model_name, car_var, fe_name, "contextual_qual")] = ctx_qual_model

print("Done. Regression files saved in:", output_dir)

In [ ]:
# ============================================================
# LaTeX regression table export
# Keeps the old regression block untouched
# ============================================================

from statsmodels.iolib.summary2 import summary_col

latex_output_dir = DATA_PROCESSED / "regression_tables_latex"
latex_output_dir.mkdir(parents=True, exist_ok=True)


def make_latex_regression_table(models, car_var, fe_name, model_name):
    """
    Creates one LaTeX regression table with five specifications:
    (1) LLM FLI with controls only
    (2) LLM FLI and contextual dimensions with controls and fixed effects
    (3) Additive LLM dimensions with dictionary FLI, without qualitative variables
    (4) Additive LLM dimensions with grouped qualitative variables
    (5) Contextualized FLI with grouped qualitative variables
    """

    info_dict = {
        "N": lambda x: f"{int(x.nobs)}",
        "R$^2$": lambda x: f"{x.rsquared:.3f}",
        "Adj. R$^2$": lambda x: f"{x.rsquared_adj:.3f}"
    }

    table = summary_col(
        models,
        stars=True,
        float_format="%0.4f",
        model_names=["(1)", "(2)", "(3)", "(4)", "(5)"],
        info_dict=info_dict
    )

    df_table = table.tables[0].copy()

    # --------------------------------------------------------
    # Always keep numerical/core variables
    # --------------------------------------------------------

    always_keep = [
        "Intercept",
        "forward_looking_intensity",
        "contextual_fli",
        "dict_score",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "bm",
        "log_word_count",
        "log_assets",
        "C(year)"
    ]

    # --------------------------------------------------------
    # Detect significant categorical variables
    # --------------------------------------------------------

    significant_categoricals = set()

    for model in models:

        for var, pval in model.pvalues.items():

            is_categorical = (
                var.startswith("C(main_focus_grouped)") or
                var.startswith("C(managerial_horizon_grouped)")
            )

            if is_categorical and pval < 0.10:
                significant_categoricals.add(var)

    # --------------------------------------------------------
    # Final keep logic
    # --------------------------------------------------------

    keep_rows = []

    i = 0

    while i < len(df_table.index):

        idx = str(df_table.index[i])

        keep = (
            any(idx.startswith(prefix) for prefix in always_keep)
            or idx in significant_categoricals
        )

        if keep:

            keep_rows.append(i)

            # Keep SE row
            if i + 1 < len(df_table.index):

                next_idx = str(df_table.index[i + 1])

                if next_idx.strip() == "":
                    keep_rows.append(i + 1)

            i += 2

        else:
            i += 1

    table.tables[0] = df_table.iloc[keep_rows].copy()

    # --------------------------------------------------------
    # Re-add model statistics rows
    # --------------------------------------------------------

    stats_rows = [
        "N",
        "R$^2$",
        "Adj. R$^2$"
    ]

    stats_df = df_table.loc[
        [idx for idx in df_table.index if idx in stats_rows]
    ].copy()

    table.tables[0] = pd.concat(
        [table.tables[0], stats_df]
    )

    # --------------------------------------------------------
    # Rename variables for thesis-style table
    # --------------------------------------------------------

    rename_dict = {
        "Intercept": "Constant",
        "forward_looking_intensity": "LLM FLI",
        "contextual_fli": "Contextualized FLI",
        "dict_score": "Dictionary FLI",
        "specificity": "Specificity",
        "economic_substance": "Economic Substance",
        "tone": "Tone",
        "certainty": "Certainty",
        "bm": "Book-to-Market",
        "log_word_count": "Log Word Count",
        "log_assets": "Log Assets"
    }

    table.tables[0].rename(index=rename_dict, inplace=True)

    # --------------------------------------------------------
    # Add specification indicator rows
    # --------------------------------------------------------

    fe_rows = pd.DataFrame(
        {
            "(1)": ["No",  "Yes", fe_name,    "No",  "No",  "No"],
            "(2)": ["No",  "Yes", fe_name, "No",  "No",  "No"],
            "(3)": ["Yes", "Yes", fe_name, "No",  "No",  "No"],
            "(4)": ["Yes", "Yes", fe_name, "Yes", "Yes", "No"],
            "(5)": ["Yes", "Yes", fe_name, "Yes", "Yes", "Yes"]
        },
        index=[
            "Dictionary Baseline Included",
            "Year FE",
            "Fixed Effects",
            "Main Focus Dummies",
            "Managerial Horizon Dummies",
            "Contextualization of LLM FLI"
        ]
    )

    table.tables[0] = pd.concat([table.tables[0], fe_rows])

    # --------------------------------------------------------
    # Convert to LaTeX
    # --------------------------------------------------------

    latex = table.as_latex()
    
    latex = latex.replace(
    "Adj. R\\$\\textasciicircum \\{2\\}\\$",
    "Adj. R\\$^2\\$ \\\\\n\\midrule"
)

    caption = (
        f"Condensed regression results for "
        f"{car_var.replace('_', ' ')} "
        f"using {model_name.capitalize()} "
        f"with {fe_name.lower()} fixed effects"
)

    label = f"tab:reg_{car_var}_{fe_name.lower()}_{model_name}"

    latex = latex.replace(
        "\\begin{table}",
        "\\begin{table}[htbp]\n\\centering"
    )

    latex = latex.replace(
        "\\caption{}",
        f"\\caption{{{caption}}}\n\\label{{{label}}}"
    )

    latex = latex.replace("\\end{table}\n","")

    latex += (
        "\n\\vspace{0.2cm}\n"
        "\\begin{minipage}{0.95\\textwidth}\n"
        "\\footnotesize\n"
        "\\textit{Notes:} This table reports OLS regression results. "
        "Standard errors are reported in parentheses and clustered by firm (PERMNO). "
        "Firm or industry fixed effects are included where indicated but not reported. "
        "Grouped qualitative LLM categories are reported where included. "
        "$^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.\n"
        "\\end{minipage}\n"
        "\\end{table}\n"
    )

    file_name = f"regression_condensed_{car_var}_{fe_name}_{model_name}.tex"
    file_path = latex_output_dir / file_name

    with open(file_path, "w") as f:
        f.write(latex)

    return latex, file_path


# ============================================================
# Export LaTeX tables from existing results dictionary
# ============================================================

for model_name in ["llama", "qwen"]:
    for car_var in car_vars:
        for fe_name in ["FIRM", "IND"]:

            models = [
                results[(model_name, car_var, fe_name, "fli_controls_only")],
                results[(model_name, car_var, fe_name, "fli_only")],
                results[(model_name, car_var, fe_name, "additive_noqual")],
                results[(model_name, car_var, fe_name, "additive_qual")],
                results[(model_name, car_var, fe_name, "contextual_qual")]
            ]

            latex, path = make_latex_regression_table(
                models=models,
                car_var=car_var,
                fe_name=fe_name,
                model_name=model_name
            )

            print("Saved:", path)